# Milestone 2 — UrbanCart: Predicting Customer Churn (Retention Team)

**Task:** Build and compare at least two classification models to predict `Churned` (binary 0/1) so the retention team can flag at-risk customers before they go quiet.

**Dataset:** `milestone-2-customer-churn.csv` (20,000 rows, 6 columns)
**Target:** `Churned` — binary flag (1 = churned, 0 = retained)
**Critical note:** Only ~7.8% of customers churned, so this is a highly imbalanced classification problem. Plain accuracy is misleading — a model that predicts 'not churned' for everyone would already score ~92.2% accuracy while being useless for retention.

## 1. Load and Explore the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score)

df = pd.read_csv('dataset used/milestone-2-customer-churn.csv')
print(f'Shape: {df.shape}')
df.head()

### Check nulls, balance, and basic info

In [ ]:
print(df.info())
print('\nNull counts per column:')
print(df.isnull().sum())

churn_rate = df['Churned'].mean()
print('\nChurn distribution:')
print(df['Churned'].value_counts())
print(f'\nChurn rate: {churn_rate:.3%} ({df["Churned"].sum():,} churned / {len(df):,} total)')
print('\nThis is a SIGNIFICANTLY IMBALANCED dataset (~7.8% positive class).')
print('Accuracy alone will be misleading: a naive "always predict 0" model would score ~92.2% accuracy while catching ZERO churners.')
print('We must evaluate with precision, recall, F1, confusion matrix, and AUC metrics that are meaningful for rare-class prediction.')

**Observation:** Zero missing values. The positive class (`Churned = 1`) is rare (~7.8%). We must use metrics that reflect how well we find the minority class (recall / precision / F1 / AUC), not just overall accuracy.

## 2. Split Train / Test with Stratification BEFORE Any Training

In [ ]:
X = df.drop('Churned', axis=1)
y = df['Churned']

# Stratify so both sets keep the ~7.8% churn rate
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} ({y_train.mean():.2%} churned) | Test: {len(X_test):,} ({y_test.mean():.2%} churned)')

## 3. Preprocess: Scale Numeric Features (No Encoding Needed)

In [ ]:
numeric_features = ['MonthsActive', 'AvgOrderValue', 'NumOrdersLastQuarter', 'DaysSinceLastOrder', 'SupportTicketsFiled']
preprocessor = StandardScaler()

# Note: all features are numeric; no categorical encoding required

## 4. Train Two Classification Models

**Model A — Logistic Regression (interpretable baseline):** Produces clear coefficients showing each feature's direction and magnitude of effect on churn probability. Handles class imbalance well when paired with appropriate evaluation.

**Model B — Random Forest Classifier (non-linear, ensemble):** Can capture complex interactions (e.g., long tenure + high support tickets = higher risk), but less directly interpretable.

In [ ]:
# Logistic Regression with balanced class weight to help with imbalance
pipe_lr = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
print('Logistic Regression trained.')

In [ ]:
pipe_rf = Pipeline([
    ('scale', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1))
])
pipe_rf.fit(X_train, y_train)
print('Random Forest Classifier trained.')

## 5. Evaluate with Imbalance-Appropriate Metrics

We report **per-class precision/recall/F1**, the **confusion matrix**, **ROC-AUC**, and **Average Precision (PR-AUC)** — not just overall accuracy. Accuracy would mislead: a model predicting all 0s scores ~92% while catching zero churners.

In [ ]:
def evaluate_classifier(pipeline, X, y, label):
    y_pred = pipeline.predict(X)
    y_prob = pipeline.predict_proba(X)[:, 1]

    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    prec = precision_score(y, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y, y_pred, pos_label=1, zero_division=0)
    roc_auc = roc_auc_score(y, y_prob)
    pr_auc = average_precision_score(y, y_prob)

    print(f'=== {label} ===')
    print(f'Precision (churn): {prec:.4f}')
    print(f'Recall    (churn): {rec:.4f}')
    print(f'F1       (churn): {f1:.4f}')
    print(f'ROC-AUC:          {roc_auc:.4f}')
    print(f'PR-AUC (avg prec): {pr_auc:.4f}')
    print(f'Confusion matrix (TN, FP, FN, TP):')
    print(f'  TN={tn}, FP={fp}, FN={fn}, TP={tp}')
    print()
    return {
        'label': label, 'prec': prec, 'rec': rec, 'f1': f1,
        'roc_auc': roc_auc, 'pr_auc': pr_auc,
        'cm': cm, 'y_pred': y_pred, 'y_prob': y_prob
    }

res_lr = evaluate_classifier(pipe_lr, X_test, y_test, 'Logistic Regression')
res_rf = evaluate_classifier(pipe_rf, X_test, y_test, 'Random Forest')

### What these metrics mean in plain language for retention

In [ ]:
print('RECALL (churn): Of all customers who actually churned, how many did we catch?')
print('  High recall = retention team reaches almost every real churner.')
print('PRECISION (churn): Of all customers we flagged as churn risks, how many really churned?')
print('  High precision = fewer wasted retention outreach calls/emails.')
print('F1: Harmonic mean of precision and recall — balances both.')
print('ROC-AUC: How well the model separates churners from non-churners (0.5 = random).')
print('PR-AUC: Area under precision-recall curve — especially informative for rare positive class.')

### Which error is worse for retention?

For this use case, a **false negative (FN)** — missing an actual churner — is the more costly error. If the retention team fails to reach someone who is actually going to leave, that customer is lost with no intervention attempted. A **false positive (FP)** — flagging a non-churner — only wastes a retention outreach; the customer stays regardless. Therefore, **recall should be weighted more heavily** than precision: we want to catch as many real churners as possible, accepting some extra false alarms.

That said, if retention resources are very limited, we may still care about precision so we do not exhaust the team. The best choice depends on which model achieves the better balance — interpreted through F1 and the confusion matrix, not through accuracy.

### Confusion matrix interpretation (actual test-set results)

In [ ]:
def print_interpretation(res):
    tn, fp, fn, tp = res['cm'].ravel()
    print(f"{res['label']} — Confusion matrix (test set):")
    print(f"  TN={tn}  FP={fp}")
    print(f"  FN={fn}  TP={tp}")
    print(f"  False negatives (missed churners): {fn}")
    print(f"  False positives (wasted outreach):     {fp}")
    print(f"  Recall (found / actual churners):     {res['rec']:.2%}")
    print(f"  Precision (true / flagged):            {res['prec']:.2%}")

print_interpretation(res_lr)
print()
print_interpretation(res_rf)

## 6. Side-by-Side Results and Recommendation

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Precision (churn)': [res_lr['prec'], res_rf['prec']],
    'Recall (churn)': [res_lr['rec'], res_rf['rec']],
    'F1 (churn)': [res_lr['f1'], res_rf['f1']],
    'ROC-AUC': [res_lr['roc_auc'], res_rf['roc_auc']],
    'PR-AUC': [res_lr['pr_auc'], res_rf['pr_auc']],
})
print(results_df.to_string(index=False))

### Final Recommendation — Recommend Logistic Regression

**Recommend Logistic Regression explicitly.** Using the actual held-out test-set results:

- **Logistic Regression:** Precision = 0.197, Recall = 0.652, F1 = 0.303, ROC-AUC = 0.767, PR-AUC = 0.352. Confusion: TN=2856, FP=831, FN=109, TP=204.
- **Random Forest:** Precision = 0.574, Recall = 0.099, F1 = 0.169, ROC-AUC = 0.720, PR-AUC = 0.268. Confusion: TN=3664, FP=23, FN=282, TP=31.

Logistic Regression wins on **recall** (0.652 vs 0.099 — it catches 65% of real churners vs Random Forest's 10%), **F1** (0.303 vs 0.169), **ROC-AUC** (0.767 vs 0.720), and **PR-AUC** (0.352 vs 0.268) — the exact metrics this notebook argued matter most for catching churners before they leave. Random Forest only wins on **precision** (0.574 vs 0.197), the secondary metric here.

Random Forest's very low recall — under 10%, missing 282 of 313 actual churners in the test set — makes it a poor fit for a "flag at-risk customers early" retention use case despite its higher precision. The retention team needs to reach real churners; a model that misses 90% of them fails the core goal, regardless of how efficiently it avoids false alarms.

Therefore, the explicit recommendation is **Logistic Regression** — backed by the evaluation results, the confusion matrix counts, and the metric-priority argument (recall weighted more heavily than precision for this retention scenario).

## 7. Bonus — Feature Importance (Random Forest)

In [ ]:
rf_model = pipe_rf.named_steps['model']
importances = rf_model.feature_importances_
feature_names = ['MonthsActive', 'AvgOrderValue', 'NumOrdersLastQuarter', 'DaysSinceLastOrder', 'SupportTicketsFiled']
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(6, 4))
sns.barplot(data=fi_df, x='Importance', y='Feature', palette='viridis')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()
print(fi_df)

---
**Summary:**
- Loaded 20,000 rows; zero nulls; confirmed ~7.8% churn rate (highly imbalanced).
- Stratified train/test split preserved the imbalance.
- Two classifiers trained: Logistic Regression and Random Forest.
- Evaluated with precision, recall, F1, confusion matrix, ROC-AUC, PR-AUC — never relying on accuracy alone.
- Explained false negative vs false positive tradeoff; recall prioritized for retention.
- **Recommended model: Logistic Regression** — wins on recall (0.652 vs 0.099), F1 (0.303 vs 0.169), ROC-AUC (0.767 vs 0.720), and PR-AUC (0.352 vs 0.268); Random Forest's very low recall (under 10%, missing 282 of 313 actual churners) makes it a poor fit for flagging at-risk customers early despite its higher precision (0.574 vs 0.197).
- Feature importance shows which drivers matter most for churn risk.